In [ ]:
#--------------------------------
# 0. Install dependencies
# 
# Note: You can run the following commands in your terminal to set up the environment.

#!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
#!python get-pip.py
#!python -m pip install --upgrade pip
#!python -m pip install scikit-learn numpy pandas matplotlib seaborn joblib
#!python -m pip install clearml

# 2.2 set clearml environment variables for windows tasking
# 윈도우의 경우에는 아래의 0.1의 과정을 수행하여 주십시요.


In [ ]:
# ------------------------------------------------
# 0.1 Initialize environment variables for windows kernel
# 커널이 윈도우인 경우에 초기화 하는 코드 입니다.
# 한 번만 호출하여 주십시요.
import os
import sys
if sys.platform == "win32":
    # 1.1 set pythonpath to import from parent directory
    parent_dir = os.path.dirname(os.getcwd())
    if os.environ.get("PYTHONPATH") is None:
        os.environ["PYTHONPATH"] = ""
    if parent_dir not in sys.path:
        sys.path.append(parent_dir)
        os.environ["PYTHONPATH"] = os.environ["PYTHONPATH"] + ";" + parent_dir
    python_def_lib_path = os.path.join(parent_dir, "venv\\Lib\\site-packages")
    if python_def_lib_path not in sys.path:
        sys.path.append(python_def_lib_path)
        os.environ["PYTHONPATH"] = os.environ["PYTHONPATH"] + ";" + python_def_lib_path
    !python -m pip install -r {parent_dir}\requirements-win.txt
    os.environ["CLEARML_WEB_HOST"] = "http://172.16.8.168:8080/"
    os.environ["CLEARML_API_HOST"] = "http://172.16.8.168:8008/"
    os.environ["CLEARML_FILES_HOST"] = "http://172.16.8.168:8081"
    os.environ["CLEARML_API_ACCESS_KEY"] = "SIJ8V8YP9PL25YEAVAWGP9NV11TMOQ"
    os.environ["CLEARML_API_SECRET_KEY"] = "6x_MEHvgrUv9-TFFDrE9BE7vb3JYOXDWq4kQtqbd58nQ5pqHEGMt6qQXxF7_HCyzq7E"


In [52]:
#-------------------------------
# 1. import libraries
import os
import sys
import platform
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib
from clearml import Task, OutputModel
__file__ = os.path.join(os.getcwd(), "standard_process.ipynb")

In [53]:
#-------------------------------
# 2. read environment variables
project_name = os.environ.get("PROJECT_NAME", "standard_process")
task_name = f"{project_name}_task"
algorithm_str = os.environ.get("ALGORITHM", "RandomForest")
parameter_list = os.environ.get("PARAMETERS", "n_estimators=100,max_depth=5").split(",")
ext_module_name = f"{os.environ.get("EXT_MODULE_NAME", "sample")}"
run_mode = os.environ.get("RUN_MODE", "debug")
data_loader_str = os.environ.get("DATA_LOADER", "sample_iris")


ext_module_lists = ext_module_name.split(",") if ext_module_name else []
parameters = {}
for parameter in parameter_list:
    key_value = parameter.split("=")
    if len(key_value) == 2:
        key, value = key_value
        parameters[key] = int(value) if value.isdigit() else value


from lib.modules import load_all_dynamic_module
class_cols = load_all_dynamic_module()
print(f'class_cols = {class_cols}')

In [54]:
# 2.1 load external module
from lib.types import RunMode
from lib.path_utils import get_home_path_and_path_splitter
from lib.module import load_all_dynamic_module

ext_modules = load_all_dynamic_module(run_mode)

load_all_dynamic_module in debug mode
#00 current_path = d:\work\python\mlops_jupyter_nodebooks\lib\path_utils.py
#01 parent_dir = d:\work\python\mlops_jupyter_nodebooks\lib
home = d:\work\python\mlops_jupyter_nodebooks, path_splitter = \
#00 current_path = d:\work\python\mlops_jupyter_nodebooks\lib\path_utils.py
#01 parent_dir = d:\work\python\mlops_jupyter_nodebooks\lib
home = d:\work\python\mlops_jupyter_nodebooks, path_splitter = \
home = d:\work\python\mlops_jupyter_nodebooks, path_splitter = \
load_class_module in debug mode
module_path_name = d:\work\python\mlops_jupyter_nodebooks\lib\plugins\sample.py
ModuleSpec(name='classname', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001E68ABBE450>, origin='d:\\work\\python\\mlops_jupyter_nodebooks\\lib\\plugins\\sample.py') <class '_frozen_importlib.ModuleSpec'>


In [55]:
## debug code
print('ext_modules', ext_modules)

ext_modules {'sample': <class 'classname.sample'>}


In [ ]:
#--------------------------------
# 3. Initialize Clearml Task
print(f"CMLEARML_API_HOST: {os.environ.get('CLEARML_API_HOST')}")
task = Task.init(project_name=project_name, task_name=task_name,
                 auto_connect_frameworks={'detect_repository': False},
                 script_path=os.getcwd(),
                 auto_resource_monitoring=True)
task.set_script(source_file=__file__)
task.force_store_standalone_script(True)
task.set_script(
    source_code="import os\n"
                "import sys\n"
                "import platform\n"
                "from sklearn.ensemble import RandomForestClassifier\n"
                "from sklearn.svm import SVC\n"
                "from sklearn.metrics import accuracy_score\n",
    filename="standard_process.py"
)
task.execute_remotely(queue_name="default", clone=False)


CMLEARML_API_HOST: http://172.16.8.168:8008/


In [57]:
from lib.data_loader.sample_lstm_ae import _data_load as data_loader_sample_lstm_ae
from lib.data_loader.sample_iris import _data_load as data_loader_sample_iris

data_loaders = {"sample_lstm_ae": data_loader_sample_lstm_ae,
                 "sample_iris": data_loader_sample_iris}

if data_loader_str not in data_loaders:
    data_loader_str = "sample_iris"
X_train, X_test, y_train, y_test = data_loaders[data_loader_str]()

In [58]:
# -------------------------------
# 5. preprocess with external module
# -------------------------------

for ext_module_name in ext_module_lists:
    result = ext_modules[ext_module_name].__pre__()
    if result is not None:
        print(f"{result}")

call pre process


In [59]:
# -------------------------------
# 6. Model training
# -------------------------------
from lib.trains.random_forest import _train as rf_train
from lib.trains.svc import _train as svc_train
if sys.platform != "win32":
    from lib.trains.lstm_ae_tensor import _train as lstmae_tensor_train
else:
    lstmae_tensor_train = None
from lib.trains.lstm_ae_pytorch import _train as lstmae_pytorch_train

algorithms = {"RandomForest": rf_train, "SVM": svc_train, "default": rf_train}

if algorithm_str not in algorithms:
    train_fnc = algorithms["default"]
else:
    train_fnc = algorithms[algorithm_str]

print(parameters)

if train_fnc != None:
    trained_model = train_fnc(X_train=X_train, y_train=y_train, parameters=parameters)
else:
    trained_model = None


{'n_estimators': 100, 'max_depth': 5}


In [60]:
# -------------------------------
# 7. Evaluate model
# -------------------------------
y_pred = trained_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 1.0


In [61]:
# -------------------------------
# 8. Save model
# 9. Upload model artifact to Clearml
# -------------------------------
model_name = f"{project_name}_model"
model_path = f"{model_name}.pt"
joblib.dump(trained_model, model_path)
output_model = OutputModel(task=task, name=model_name, framework="sklearn")
output_model.update_weights(weights_filename=model_path)
task.upload_artifact(name=model_name, artifact_object=model_path)
print(f"Model saved and uploaded to Clearml: {model_path}")


Model saved and uploaded to Clearml: standard_process_model.pt


In [62]:
# -------------------------------
# 10. End of process
task.close()